# Utilities

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np

RANDOM_SEED = 42

SUBTYPES = ['ccRCC', 'pRCC', 'CHROMO', 'ONCOCYTOMA']
INFO = ['Non-Tumor', 'Uncertain (NT)', 'Uncertain (T)', 'node_count_1', 'node_count_2']

LEAF1_SUBTYPES = SUBTYPES[2:4]
LEAF2_SUBTYPES = SUBTYPES[0:2]

# Confidence Interval
N_BOOT = 10000

def bootstrap_ci_accuracy(y_true, y_pred, n_boot=N_BOOT, seed=RANDOM_SEED):

    rng = np.random.default_rng(seed)
    n = len(y_true)

    point = accuracy_score(y_true, y_pred)

    boots = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[b] = accuracy_score(y_true[idx], y_pred[idx])

    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi


from sklearn.metrics import accuracy_score
import numpy as np

def cv_accuracy_std(gp_dict, pred_col, true_col='subtype'):
    fold_acc = []

    for fold_df in gp_dict.values():
        acc = accuracy_score(fold_df[true_col], fold_df[pred_col])
        fold_acc.append(acc)

    mean_acc = np.mean(fold_acc)
    std_acc = np.std(fold_acc, ddof=1)

    return mean_acc, std_acc, fold_acc

def plot_confusion_matrix(y_true, y_pred, figsize=(10, 10),
                          title=None, save_path=None, limit=(0, 5, 1),
                          n_boot=N_BOOT, seed=RANDOM_SEED):
    labels = SUBTYPES
    display_labels = ['ccRCC', 'pRCC', 'chRCC', 'ONCO']
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_sum = np.sum(cm, axis=1, keepdims=True)
    cm_perc = cm / cm_sum.astype(float) * 100
    annot = np.empty_like(cm).astype(str)
    nrows, ncols = cm.shape

    for i in range(nrows):
        for j in range(ncols):
            c = cm[i, j].item()
            p = cm_perc[i, j].item()
            if i == j:
                s = cm_sum[i].item()
                annot[i, j] = '%.1f%%\n%d/%d' % (p, c, s)
            elif c == 0:
                annot[i, j] = ''
            else:
                annot[i, j] = '%.1f%%\n%d' % (p, c)

    cm_df = pd.DataFrame(cm, index=display_labels, columns=display_labels)
    cm_df.index.name = 'Actual'
    cm_df.columns.name = 'Predicted'
    _, ax = plt.subplots(figsize=figsize)
    heatmap = sns.heatmap(cm_df, cmap="binary", annot=annot, fmt='', ax=ax, 
                linecolor='black', linewidths=0.2, cbar=True,
                annot_kws={"size": 14}, vmin=limit[0], vmax=limit[1])
    
    colorbar = heatmap.collections[0].colorbar
    colorbar.set_ticks(np.arange(limit[0], limit[1]+1, limit[2]))
    colorbar.ax.tick_params(labelsize=14)

    ax.set_xlabel('Predicted', fontsize=14)
    ax.set_ylabel('Actual', fontsize=14)
    ax.tick_params(axis='both', labelsize=12)

    if title:
        ax.set_title(title)
        print(f"# {title}\n")

    print("# Class       | Sensitivity (Recall) | Specificity | Precision | F1 Score")
    print("# -----------------------------------------------------------------------")

    for i, label in enumerate(display_labels):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        precision   = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1          = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

        print(f"# {label:<11}| {sensitivity:>8.2f}               | {specificity:>8.4f}    | {precision:>7.3f}   | {f1:>7.3f}")

    accuracy, lo_acc, hi_acc = bootstrap_ci_accuracy(y_true, y_pred, n_boot, seed)
    print(f"\n# Overall Accuracy: {accuracy:.4f} - ({lo_acc:.4f} - {hi_acc:.4f})\n")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        
    plt.show()

def subtype_majority(row):
    counts = row[SUBTYPES].to_dict()
    majority_vote = max(counts, key=counts.get)
    return majority_vote

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

model_grids = {
    'DecisionTree': (DecisionTreeClassifier, {
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    }),
    'RandomForest': (RandomForestClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    'XGBoost': (XGBClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1]
    }),
    'ExtraTrees': (ExtraTreesClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    'GradientBoosting': (GradientBoostingClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.01, 0.1]
    }),
    'SVM': (SVC, {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf']
    }),
    'LogisticRegression': (LogisticRegression, {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'max_iter': [100, 200]
    }),
    'KNN': (KNeighborsClassifier, {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    })
}

# Histology-only (MCExpertDT) Analysis

#### Load Data

In [ ]:
import pandas as pd
import numpy as np
import json
import os

try:
    with open("_info/config.json", 'r') as file:
        conf = json.load(file)
except:
    raise FileNotFoundError("Couldn't Load Config File. Check if the `config.json` file exists under `_info` directory.")

mcd = {}

eval_type = 'val'
mcd_fn = {'Fold1':'runtime_Fold1', 'Fold2':'runtime_Fold2', 'Fold3':'runtime_Fold3', 'Fold4':'runtime_Fold4'}

for fold, filename in mcd_fn.items():
    mcd[fold] = pd.read_csv(os.path.join(conf['results_dir'],
                                         eval_type,
                                         'preds',
                                         f"{filename}.csv"))

#### Get Patient-level Aggregation

In [ ]:
gp_mcd = {}

for fold, fold_df in mcd.items():

    most_common_subtype = fold_df.groupby('id')['subtype'].agg(lambda x: x.mode()[0] if not x.mode().empty else None).reset_index()
    gp_mcd[fold] = fold_df.groupby('id')[SUBTYPES+INFO].sum().reset_index()
    gp_mcd[fold] = pd.merge(gp_mcd[fold], most_common_subtype, on='id', how='left')
    gp_mcd[fold]['result'] = gp_mcd[fold].apply(subtype_majority, axis=1)
    gp_mcd[fold]['leaf_1'] = gp_mcd[fold][LEAF1_SUBTYPES].sum(axis=1)
    gp_mcd[fold]['leaf_2'] = gp_mcd[fold][LEAF2_SUBTYPES].sum(axis=1)
    gp_mcd[fold]['correct'] = (gp_mcd[fold]['subtype']==gp_mcd[fold]['result']).map({False: 'Incorrect', True: 'Correct'})

all_subtypes = np.concatenate([fold_gp['subtype'] for fold_gp in gp_mcd.values()])
all_results = np.concatenate([fold_gp['result'] for fold_gp in gp_mcd.values()])

#### Plot Conusion Matrix

In [ ]:
mean_acc, std_acc, fold_acc = cv_accuracy_std(gp_dict=gp_mcd,
                                              pred_col='result',
                                              true_col='subtype')

print(f"# Std: {std_acc}")
plot_confusion_matrix(all_subtypes, 
        all_results, 
        figsize=(6, 5), 
        title="",
        limit=(0, 20, 5))

# Post-IHC Analysis

**NOTE:** Pre-IHC has been reported in previous section. Run previous cells and continue.

#### Get Confidence

In [ ]:
import pandas as pd
import plotly.express as px

gp_cert = {k: v.copy(deep=True) for k, v in gp_mcd.items()}

for fold, fold_gp in gp_cert.items():
    fold_gp['confidence'] = (fold_gp[SUBTYPES].max(axis=1) / fold_gp[SUBTYPES].sum(axis=1))

gp_combined_cert = pd.concat(
    [df.assign(fold=fold) for fold, df in gp_cert.items()],
    ignore_index=True
)

gp_combined_all = gp_combined_cert.copy()
gp_combined_all["fold"] = "All"

gp_combined_full = pd.concat([gp_combined_cert, gp_combined_all], ignore_index=True)

fig = px.box(
    gp_combined_full,
    x="fold",
    y="confidence",
    color="correct",
    boxmode='group',
    points='outliers',
    title="Confidence in Predictions (Per Fold and Overall)",
    labels={"confidence": "Confidence", "fold": "Fold", "correct": "Prediction"}
)

fig.update_layout(
    width=1000,
    height=500,
    legend_title="Prediction"
)

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D

# Filter only the "All" fold
all_df = gp_combined_full[gp_combined_full["fold"] == "All"].copy()

# Color map for groups
color_map = {'Correct': 'green', 'Incorrect': 'darkred'}

# Setup plot
fig, ax = plt.subplots(figsize=(4, 6))

sns.boxplot(
    y='confidence',          
    x='correct',
    data=all_df,
    palette=color_map,
    width=0.3,
    linewidth=1.5,
    order=['Incorrect', 'Correct']
)

ax.set_ylabel(r"Confidence ($C_p$)", fontsize=16)
ax.set_xlabel('Subtype Classification', fontsize=16)
ax.set_yticks(np.arange(0, 1.05, 0.1))
ax.tick_params(axis='both', labelsize=14)


plt.tight_layout()
plt.show()

incorrect_df = all_df[all_df['correct'] == 'Incorrect']
q3_incorrect = incorrect_df['confidence'].quantile(0.75)

print("Q3 (Incorrect):", q3_incorrect)

correct_df = all_df[all_df['correct'] == 'Correct']
q1_correct = correct_df['confidence'].quantile(0.25)
print("Q1 (Correct):", q1_correct)

#### Plot Uncertainty Metrics for Different Thresholds

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

all_df = pd.concat([df.copy() for df in gp_cert.values()], ignore_index=True)

CERT_THRESHOLD = 0.80

threshold_range = np.linspace(CERT_THRESHOLD - 0.2, CERT_THRESHOLD + 0.2, 100)
metrics = {'Rcc': [], 'Riu': [], 'UA': []}
counts = {'Nu': [], 'Ncc': [], 'Nic': [], 'Niu': [], 'Ncu': []}
range_thr = []

for threshold in threshold_range:
    all_df['confident'] = all_df['confidence'] >= threshold

    N = len(all_df)
    Nu = (~all_df['confident']).sum()
    Ncc = ((all_df['correct'] == 'Correct') & all_df['confident']).sum()
    Nic = ((all_df['correct'] == 'Incorrect') & all_df['confident']).sum()
    Niu = ((all_df['correct'] == 'Incorrect') & ~all_df['confident']).sum()
    Ncu = ((all_df['correct'] == 'Correct') & ~all_df['confident']).sum()

    Rcc = Ncc / (Ncc + Nic) if (Ncc + Nic) > 0 else 0
    Riu = Niu / (Niu + Nic) if (Niu + Nic) > 0 else 0
    UA = (Ncc + Niu) / N if N > 0 else 0

    # Store results
    metrics['Rcc'].append(Rcc)
    metrics['Riu'].append(Riu)
    metrics['UA'].append(UA)

    counts['Nu'].append(Nu)
    counts['Ncc'].append(Ncc)
    counts['Nic'].append(Nic)
    counts['Niu'].append(Niu)
    counts['Ncu'].append(Ncu)

    range_thr.append(threshold)

# Plot
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[
                        'Metrics vs. Confidence Threshold (All Folds)',
                        'Counts vs. Confidence Threshold (All Folds)'
                    ])

# Metrics
for metric, values in metrics.items():
    fig.add_trace(go.Scatter(x=range_thr, y=values, mode='lines+markers', name=metric), row=1, col=1)

# Counts
for count_metric, values in counts.items():
    fig.add_trace(go.Scatter(x=range_thr, y=values, mode='lines+markers', name=count_metric), row=2, col=1)

# Layout
fig.update_layout(height=650, width=900, title_text="Uncertainty-Aware Metrics across Thresholds")
fig.update_xaxes(title_text="Confidence Threshold", row=2, col=1)
fig.update_yaxes(title_text="Metric Value", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)

fig.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

all_df = pd.concat([df.copy() for df in gp_cert.values()], ignore_index=True)
threshold_range = np.linspace(CERT_THRESHOLD-0.2, CERT_THRESHOLD+0.2, 100)

thresholds = []
Nu_list, Ncc_list, Nic_list, Niu_list, Ncu_list = [], [], [], [], []

for threshold in threshold_range:
    all_df['confident'] = all_df['confidence'] >= threshold

    Nu = (~all_df['confident']).sum()
    Ncc = ((all_df['correct'] == 'Correct') & all_df['confident']).sum()
    Nic = ((all_df['correct'] == 'Incorrect') & all_df['confident']).sum()
    Niu = ((all_df['correct'] == 'Incorrect') & ~all_df['confident']).sum()
    Ncu = ((all_df['correct'] == 'Correct') & ~all_df['confident']).sum()

    thresholds.append(threshold)
    Nu_list.append(Nu)
    Ncc_list.append(Ncc)
    Nic_list.append(Nic)
    Niu_list.append(Niu)
    Ncu_list.append(Ncu)

fig, ax = plt.subplots(figsize=(6, 6)) 

MARKER_SIZE = 3
LINE_WIDTH = 1

ax.plot(thresholds, Ncc_list, label=r'$N_{cc}$', color='green', marker='o', markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
ax.plot(thresholds, Nic_list, label=r'$N_{ic}$', color='red', marker='o', markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
ax.plot(thresholds, Ncu_list, label=r'$N_{cu}$', color='yellowgreen', marker='o', markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
ax.plot(thresholds, Niu_list, label=r'$N_{iu}$', color='orange', marker='o', markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
ax.plot(thresholds, Nu_list, label=r'$N_{u}$', color='gray', linestyle='--', marker='o', markersize=MARKER_SIZE, linewidth=LINE_WIDTH)

ax.axvline(x=CERT_THRESHOLD, color='black', linestyle='--', linewidth=1.2, label=r"$C_{\tau}$"+f" = {CERT_THRESHOLD}")

ax.set_xlim(CERT_THRESHOLD-0.2, CERT_THRESHOLD+0.2)
ax.set_ylim(0, 80)

ax.tick_params(axis='both', labelsize=14)

ax.set_xlabel(r"Confidence Threshold ($C_{\tau}$)", fontsize=16)
ax.set_ylabel("No. Patients", fontsize=16)
ax.legend(title="Metric")
ax.grid(True)
plt.tight_layout()
plt.show()

#### Post-IHC Results

##### Load IHC Data

In [ ]:
import pandas as pd
import numpy as np


NAN_VALS = []

tr_val_ihc = pd.read_csv("/path/to/ihc/_data/sample_data.csv")
ihc_gs_df = pd.read_csv("/path/to/ihc/_res/grid_search.csv")
# For test, you should also load the data related to that in both pre and post IHC
# modules

with open(f'_info/split_ids.json', 'r') as f:
    split_ids = json.load(f)

# List of Biomarkers (in order of mRMR)
BIOMARKERS = ['VIM', 'CK7', 'CA9', 'ECAD', 'P504', 'CD10']
RANDOM_SEED = 42
FINAL_THRESHOLD = 0.80

columns_to_convert = [col for col in tr_val_ihc.columns if col not in ['id', 'subtype']]
tr_val_ihc[columns_to_convert] = tr_val_ihc[columns_to_convert].apply(pd.to_numeric, errors='coerce')

p_cols = [col for col in tr_val_ihc.columns if col.endswith('_p')]
for p_col in p_cols:
    base = p_col[:-2]
    i_col = f"{base}_i"
    
    if i_col in tr_val_ihc.columns:
        q_col = f"{base}"
        tr_val_ihc[q_col] = tr_val_ihc[p_col] * tr_val_ihc[i_col]

tr_val_ihc = tr_val_ihc.drop(columns=p_cols + [f"{col[:-2]}_i" for col in p_cols])

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import ast

best_entry = ihc_gs_df.iloc[0]
best_model = best_entry['model']
best_params = ast.literal_eval(best_entry['params'])
model_class = model_grids[best_model][0]

if 'random_state' in model_class().get_params():
    model = model_class(**best_params, random_state=RANDOM_SEED)
else:
    model = model_class(**best_params)

train_dfs = {}
val_dfs = {}

for fold_name, fold_data in split_ids.items():
    
    train_ids = {subtype: set(ids) for subtype, ids in fold_data['Train'].items()}
    val_ids = {subtype: set(ids) for subtype, ids in fold_data['Test'].items()}
    
    train_mask = tr_val_ihc.apply(lambda row: row['id'] in train_ids.get(row['subtype'], set()), axis=1)
    val_mask = tr_val_ihc.apply(lambda row: row['id'] in val_ids.get(row['subtype'], set()), axis=1)
    
    train_dfs[fold_name] = tr_val_ihc[train_mask].reset_index(drop=True)
    val_dfs[fold_name] = tr_val_ihc[val_mask].reset_index(drop=True)

gp_ihc = {k: v.copy(deep=True) for k, v in gp_mcd.items()}
u_ids = {}
u_tot = 0

for fold, fold_gp in gp_ihc.items():
    fold_gp['confidence'] = (fold_gp[SUBTYPES].max(axis=1) / fold_gp[SUBTYPES].sum(axis=1))
    u_ids[fold] = gp_ihc[fold][gp_ihc[fold]['confidence']<=FINAL_THRESHOLD]['id'].to_list()
    u_tot += len(u_ids[fold])
    print(f">>> {fold}: '{len(u_ids[fold])}' Patients Flagged For IHC")

print(f"---------------------------------------------")
print(f">>> Total: '{u_tot}' Patients Flagged For IHC\n")

for i in range(1, len(BIOMARKERS) + 1):

    selected = BIOMARKERS[:i]
    print(f">>> Evaluate IHC Panel with '{i} ({selected})' Biomarkers")

    for fold, fold_gp in gp_ihc.items():

        train_data = train_dfs[fold].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        X_train_full = train_data[selected]
        y_train_full = train_data['subtype']
        train_ids = train_data['id']

        X_val = val_dfs[fold][val_dfs[fold]['id'].isin(u_ids[fold])]
        val_ids = X_val['id']
        X_val = X_val[selected]

        label_encoder = LabelEncoder()
        y_train_encoded = label_encoder.fit_transform(y_train_full)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_full)
        X_val_scaled = scaler.transform(X_val)

        model.fit(X_train_scaled, y_train_encoded)

        y_pred_val = model.predict(X_val_scaled)
        y_pred_val_encoded = label_encoder.inverse_transform(y_pred_val)

        ihc_results = pd.DataFrame({
            'id': val_ids,
            'prediction': y_pred_val_encoded
        })

        id_to_pred = ihc_results.set_index('id')['prediction']
        gp_ihc[fold][f'ihc_{i}'] = gp_ihc[fold]['id'].map(id_to_pred)
        gp_ihc[fold][f'ihc_{i}'] = gp_ihc[fold][f'ihc_{i}'].fillna(gp_ihc[fold]['result'])

In [ ]:
for i in range(1, len(BIOMARKERS) + 1):

    for fold, fold_gp in gp_ihc.items():

        col_pred = f'ihc_{i}'
        col_result = f'ihc_result_{i}'
        gp_ihc[fold][col_result] = (gp_ihc[fold][col_pred] == gp_ihc[fold]['subtype']).map({True: 'Correct', False: 'Incorrect'})

    all_subtypes = np.concatenate([fold_gp['subtype'] for fold_gp in gp_ihc.values()])
    all_results = np.concatenate([fold_gp[f'ihc_{i}'] for fold_gp in gp_ihc.values()])

    incorrect_ids = (
        pd.concat(gp_ihc.values(), ignore_index=True) 
          .loc[lambda d: d[col_result] == "Incorrect", "id"]
          .tolist()
    )
    mean_acc, std_acc, fold_acc = cv_accuracy_std(gp_dict=gp_ihc, pred_col=col_pred, true_col='subtype')

    print(f"# Std: {std_acc}")
    print(f"# Incorrect IDs ({i} Biomarkers) → {incorrect_ids}")

    plot_confusion_matrix(all_subtypes, 
        all_results, 
        figsize=(6, 5), 
        title="",
        limit=(0, 20, 5))